In [ ]:
from importlib.util import find_spec
import subprocess
import sys

missing = [name for name in ("numpy", "scipy") if find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

from pathlib import Path
from statistics import median
from time import perf_counter

import numpy as np
from scipy import sparse

rng = np.random.default_rng(42)
print("Библиотеки подключены ")

Библиотеки подключены 


In [ ]:
# задание 6.1. Срезы одномерного массива
arr = np.arange(20)
print("Каждый третий с индекса 1 ", arr[1::3])
print("Индексы с 5 до 15 (без 15) в обратном порядке ", arr[5:15][::-1])
print("Последние пять в обратном порядке ", arr[:-6:-1])

Каждый третий с индекса 1  [ 1  4  7 10 13 16 19]
Индексы с 5 до 15 (без 15) в обратном порядке  [14 13 12 11 10  9  8  7  6  5]
Последние пять в обратном порядке  [19 18 17 16 15]


In [ ]:
# задание 6.2. Срезы и диагональ матрицы
mat = rng.integers(0, 10, size=(6, 6))
print("Исходная матрица \n", mat)
print("Индексы строк 2–4 и столбцов 1–3 \n", mat[2:5, 1:4])
print("Главная диагональ ", np.diagonal(mat))
rows, cols = np.indices(mat.shape)
print("Элементы на клетках с чётной суммой индексов \n",
      mat[(rows + cols) % 2 == 0].reshape(6, 3))

Исходная матрица 
 [[0 7 6 4 4 8]
 [0 6 2 0 5 9]
 [7 7 7 7 5 1]
 [8 4 5 3 1 9]
 [7 6 4 8 5 4]
 [4 2 0 5 8 0]]
Индексы строк 2–4 и столбцов 1–3 
 [[7 7 7]
 [4 5 3]
 [6 4 8]]
Главная диагональ  [0 6 7 3 5 0]
Элементы на клетках с чётной суммой индексов 
 [[0 6 4]
 [6 0 9]
 [7 7 5]
 [4 3 9]
 [7 4 5]
 [2 5 0]]


In [ ]:
# задание 6.3. Выбор по индексам
a = np.array([10, 20, 30, 40, 50, 60])
m = np.arange(12).reshape(3, 4)
print("Элементы массива ", a[[0, 2, 4]])
print("Три пары координат ", m[[0, 1, 2], [1, 3, 0]])
print("Блок 2×2 \n", m[np.ix_([0, 2], [1, 3])])

Элементы массива  [10 30 50]
Три пары координат  [1 7 8]
Блок 2×2 
 [[ 1  3]
 [ 9 11]]


In [ ]:
# задание 6.4. Булевы маски
np.random.seed(42)
x = np.random.randn(1000)
print("Количество значений больше 1 ", np.count_nonzero(x > 1))
positive = x.copy()
positive[positive < 0] = 0
print("Замена отрицательных, первые 10 ", positive[:10])
changed = x.copy()
indices = np.flatnonzero(x[:-2] * x[2:] < 0) + 1
changed[indices] = 0
print("Исходные первые 10 ", x[:10])
print("После проверки соседей ", changed[:10])
print("Всего позиций с соседями разных знаков ", len(indices))

Количество значений больше 1  155
Замена отрицательных, первые 10  [0.49671415 0.         0.64768854 1.52302986 0.         0.
 1.57921282 0.76743473 0.         0.54256004]
Исходные первые 10  [ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337 -0.23413696
  1.57921282  0.76743473 -0.46947439  0.54256004]
После проверки соседей  [ 0.49671415 -0.1382643   0.          0.          0.          0.
  0.          0.         -0.46947439  0.54256004]
Всего позиций с соседями разных знаков  512


In [ ]:
# задание 7.1. Создание big.dat порциями
path = Path("big.dat").resolve()
shape = (50_000, 200)
dtype = np.dtype("float32")
batch_size = 1000
data = np.memmap(path, dtype=dtype, mode="w+", shape=shape)
for start in range(0, shape[0], batch_size):
    stop = min(start + batch_size, shape[0])
    data[start:stop] = rng.random((stop - start, shape[1]), dtype=dtype)
data.flush()
del data
file_size = path.stat().st_size
assert file_size == shape[0] * shape[1] * dtype.itemsize
print("Файл ", path)
print(f"Размер  {file_size:,} байт")

Файл  /content/big.dat
Размер  40,000,000 байт


In [ ]:
# задание 7.2. Минимум, максимум и среднее по батчам
data = np.memmap(path, dtype=dtype, mode="r", shape=shape)
minimum, maximum = np.inf, -np.inf
total = 0.0
for start in range(0, shape[0], batch_size):
    batch = data[start:start + batch_size]
    minimum = min(minimum, batch.min())
    maximum = max(maximum, batch.max())
    total += batch.sum(dtype=np.float64)
print("Минимум = ", minimum, "Максимум = ", maximum, "Среднее = ", total / data.size)
del batch, data

Минимум =  0.0 Максимум =  0.9999999 Среднее =  0.49993614274504183


In [ ]:
# задание 7.3. Порядок хранения и обхода
def sum_rows(arr):
    sums = []
    for i in range(arr.shape[0]):
        total = 0.0
        for j in range(arr.shape[1]):
            total += arr[i, j]
        sums.append(total)
    return sums

def sum_cols(arr):
    sums = []
    for j in range(arr.shape[1]):
        total = 0.0
        for i in range(arr.shape[0]):
            total += arr[i, j]
        sums.append(total)
    return sums

c_mat = rng.random((3000, 3000))
f_mat = np.asfortranarray(c_mat)
for order, matrix in [("C", c_mat), ("F", f_mat)]:
    for direction, function, axis in [("строки", sum_rows, 1), ("столбцы", sum_cols, 0)]:
        start = perf_counter()
        sums = function(matrix)
        elapsed = perf_counter() - start
        np.testing.assert_allclose(sums, matrix.sum(axis=axis))
        print(f"Порядок {order}, {direction}: {elapsed:.3f} с")
del c_mat, f_mat, matrix

Порядок C, строки: 3.625 с
Порядок C, столбцы: 2.437 с
Порядок F, строки: 2.397 с
Порядок F, столбцы: 2.248 с


In [ ]:
# задание 7.4. Вычисление выражения тремя способами
size = 10_000_000
a = rng.random(size)
b = rng.random(size)

expected_list = np.empty_like(a)
loop_seconds = 0.0
for start_idx in range(0, size, 100_000):
    stop_idx = min(start_idx + 100_000, size)
    a_list = a[start_idx:stop_idx].tolist()
    b_list = b[start_idx:stop_idx].tolist()
    block = []
    start = perf_counter()
    for ai, bi in zip(a_list, b_list):
        block.append((ai + bi) / (ai - bi + 1))
    loop_seconds += perf_counter() - start
    expected_list[start_idx:stop_idx] = block
print(f"Python - цикл со списками  {loop_seconds:.3f} с")
del a_list, b_list, block

start = perf_counter()
expected = (a + b) / (a - b + 1)
print(f"NumPy - одна строка  {perf_counter() - start:.3f} с")
np.testing.assert_allclose(expected_list, expected)
del expected_list

result, denominator = np.empty_like(a), np.empty_like(a)
start = perf_counter()
np.add(a, b, out=result)
np.subtract(a, b, out=denominator)
np.add(denominator, 1, out=denominator)
np.divide(result, denominator, out=result)
print(f"NumPy - готовые буферы out= {perf_counter() - start:.3f} с")
np.testing.assert_allclose(result, expected)
print("Все три результата совпадают ")
del a, b, result, denominator, expected

Python - цикл со списками  4.072 с
NumPy - одна строка  0.149 с
NumPy - готовые буферы out= 0.100 с
Все три результата совпадают 


In [ ]:
# задание 8.1. Из COO в CSR с учётом дубликатов
row = rng.integers(0, 5000, 10_000).tolist()
col = rng.integers(0, 5000, 10_000).tolist()
values = rng.integers(1, 10, 10_000).tolist()
row[-1], col[-1] = row[0], col[0]
coo = sparse.coo_matrix((values, (row, col)), shape=(5000, 5000))
csr = coo.tocsr()
unique = len(set(zip(row, col)))
assert csr.nnz == unique
print("Записей COO ", coo.nnz, "Уникальных координат = ", unique, "nnz CSR ", csr.nnz)

Записей COO  10000 Уникальных координат =  9997 nnz CSR  9997


In [ ]:
# задание 8.2. Память и время умножения на вектор
def multiply_time(matrix, vector):
    matrix @ vector
    times = []
    for _ in range(5):
        start = perf_counter()
        result = matrix @ vector
        times.append(perf_counter() - start)
    return median(times), result

def compare_matrices(n, density):
    csr = sparse.random(n, n, density=density, format="csr", dtype=np.float64,
                        random_state=rng, data_rvs=rng.random)
    dense = csr.toarray()
    vector = rng.random(n)
    dense_time, y_dense = multiply_time(dense, vector)
    csr_time, y_csr = multiply_time(csr, vector)
    np.testing.assert_allclose(y_dense, y_csr)
    csr_bytes = csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes
    return dense.nbytes, csr_bytes, dense_time, csr_time

dense_bytes, csr_bytes, dense_time, csr_time = compare_matrices(4000, 0.005)
for name, count, seconds in [("dense", dense_bytes, dense_time), ("CSR", csr_bytes, csr_time)]:
    print(f"{name}: {count:,} байт; A @ x  {seconds * 1000:.3f} мс")
print(f"В CSR данные занимают в {dense_bytes / csr_bytes:.1f} раза меньше памяти.")

dense: 128,000,000 байт; A @ x  9.666 мс
CSR: 976,004 байт; A @ x  0.203 мс
В CSR данные занимают в 131.1 раза меньше памяти.


In [ ]:
# задание 8.3. Поиск порога по таблице замеров
densities = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
first = None
print("Плотность    dense, мс    CSR, мс    Быстрее")
for density in densities:
    _, _, dense_time, csr_time = compare_matrices(2000, density)
    winner = "CSR" if csr_time < dense_time else "dense"
    print(f"{density:9.3f}    {dense_time * 1000:9.3f}    {csr_time * 1000:7.3f}    {winner}")
    if first is None and csr_time >= dense_time:
        first = density
if first is None:
    print("CSR быстрее при всех проверенных плотностях ")
else:
    print(f"Первая проверенная плотность, при которой CSR перестаёт выигрывать  {first:.1%}.")

Плотность    dense, мс    CSR, мс    Быстрее
    0.001        2.559      0.036    CSR
    0.005        6.473      0.070    CSR
    0.010        6.387      0.091    CSR
    0.050        7.056      0.511    CSR
    0.100        3.951      0.834    CSR
    0.200        5.987      5.500    CSR
    0.500        3.997      7.772    dense
    1.000        3.430     10.684    dense
Первая проверенная плотность, при которой CSR перестаёт выигрывать  50.0%.


In [ ]:
#Тема 2 Pandas

In [1]:
# //1.1 — Создание Series с собственным индексом
import pandas as pd

scores = pd.Series([85, 92, 78], index=["Анна", "Иван", "Ольга"], name="Баллы")
print(scores)

Анна     85
Иван     92
Ольга    78
Name: Баллы, dtype: int64


In [2]:
# //1.2 — Создание DataFrame из словаря
employees = {
    "name": ["Анна", "Иван", "Ольга", "Павел", "Елена", "Ирина", "Денис", "Алла", "Олег"],
    "department": ["IT", "IT", "IT", "Продажи", "Продажи", "Продажи", "HR", "HR", "Архив"],
    "salary": [95000, 75000, None, 90000, 70000, 65000, 68000, None, None],
    "bonus": [12000, None, 6000, 14000, 5000, None, 4000, 3000, None],
    "hire_date": pd.to_datetime(["2023-02-01", "2024-03-14", "2025-01-20",
                                 "2023-11-05", "2024-07-10", "2025-09-01",
                                 "2022-10-12", "2024-05-16", "2025-04-01"])
}
df = pd.DataFrame(employees)
print(df)

    name department   salary    bonus  hire_date
0   Анна         IT  95000.0  12000.0 2023-02-01
1   Иван         IT  75000.0      NaN 2024-03-14
2  Ольга         IT      NaN   6000.0 2025-01-20
3  Павел    Продажи  90000.0  14000.0 2023-11-05
4  Елена    Продажи  70000.0   5000.0 2024-07-10
5  Ирина    Продажи  65000.0      NaN 2025-09-01
6  Денис         HR  68000.0   4000.0 2022-10-12
7   Алла         HR      NaN   3000.0 2024-05-16
8   Олег      Архив      NaN      NaN 2025-04-01


In [3]:
# //1.3 — Первые и последние строки, типы и статистика
print("Первые три строки")
print(df.head(3))
print("\nПоследние три строки")
print(df.tail(3))
print("\nТипы данных и количество заполненных значений")
df.info()
print("\nСтатистика зарплат и бонусов")
print(df[["salary", "bonus"]].describe().round(2))

Первые три строки
    name department   salary    bonus  hire_date
0   Анна         IT  95000.0  12000.0 2023-02-01
1   Иван         IT  75000.0      NaN 2024-03-14
2  Ольга         IT      NaN   6000.0 2025-01-20

Последние три строки
    name department   salary   bonus  hire_date
6  Денис         HR  68000.0  4000.0 2022-10-12
7   Алла         HR      NaN  3000.0 2024-05-16
8   Олег      Архив      NaN     NaN 2025-04-01

Типы данных и количество заполненных значений
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   name        9 non-null      object        
 1   department  9 non-null      object        
 2   salary      6 non-null      float64       
 3   bonus       6 non-null      float64       
 4   hire_date   9 non-null      datetime64[ns]
dtypes: datetime64[ns](1), float64(2), object(2)
memory usage: 492.0+ bytes

Статистика з

In [4]:
# //2.1 — Имена и зарплаты сотрудников IT
print(df.loc[df["department"] == "IT", ["name", "salary"]])

    name   salary
0   Анна  95000.0
1   Иван  75000.0
2  Ольга      NaN


In [5]:
# //2.2 — Первые три строки и первые два столбца
print(df.iloc[:3, :2])

    name department
0   Анна         IT
1   Иван         IT
2  Ольга         IT


In [6]:
# //2.3 — Сотрудники с зарплатой выше 70 000
print(df.loc[df["salary"] > 70000, ["name", "salary"]])

    name   salary
0   Анна  95000.0
1   Иван  75000.0
3  Павел  90000.0


In [7]:
# //2.4 — Расчёт общей компенсации
df["total_comp"] = df["salary"] + df["bonus"].fillna(0)
print(df[["name", "salary", "bonus", "total_comp"]])

    name   salary    bonus  total_comp
0   Анна  95000.0  12000.0    107000.0
1   Иван  75000.0      NaN     75000.0
2  Ольга      NaN   6000.0         NaN
3  Павел  90000.0  14000.0    104000.0
4  Елена  70000.0   5000.0     75000.0
5  Ирина  65000.0      NaN     65000.0
6  Денис  68000.0   4000.0     72000.0
7   Алла      NaN   3000.0         NaN
8   Олег      NaN      NaN         NaN


In [8]:
# //2.5 — Удаление даты приёма
df = df.drop(columns="hire_date")
print(df)

    name department   salary    bonus  total_comp
0   Анна         IT  95000.0  12000.0    107000.0
1   Иван         IT  75000.0      NaN     75000.0
2  Ольга         IT      NaN   6000.0         NaN
3  Павел    Продажи  90000.0  14000.0    104000.0
4  Елена    Продажи  70000.0   5000.0     75000.0
5  Ирина    Продажи  65000.0      NaN     65000.0
6  Денис         HR  68000.0   4000.0     72000.0
7   Алла         HR      NaN   3000.0         NaN
8   Олег      Архив      NaN      NaN         NaN


In [9]:
# //3.1 — Количество пропусков по столбцам
print(df.isna().sum())

name          0
department    0
salary        3
bonus         3
total_comp    3
dtype: int64


In [10]:
# //3.2 — Заполнение зарплаты средним по своему отделу
department_mean = df.groupby("department")["salary"].transform("mean")
df["salary"] = df["salary"].fillna(department_mean)
print(df[["name", "department", "salary"]])

    name department   salary
0   Анна         IT  95000.0
1   Иван         IT  75000.0
2  Ольга         IT  85000.0
3  Павел    Продажи  90000.0
4  Елена    Продажи  70000.0
5  Ирина    Продажи  65000.0
6  Денис         HR  68000.0
7   Алла         HR  68000.0
8   Олег      Архив      NaN


In [11]:
# //3.3 — Заполнение пропущенных бонусов нулями
df["bonus"] = df["bonus"].fillna(0)
print(df[["name", "bonus"]])

    name    bonus
0   Анна  12000.0
1   Иван      0.0
2  Ольга   6000.0
3  Павел  14000.0
4  Елена   5000.0
5  Ирина      0.0
6  Денис   4000.0
7   Алла   3000.0
8   Олег      0.0


In [12]:
# //3.4 — Удаление строк с оставшимися пропусками в зарплате
# В отделе «Архив» нет ни одной известной зарплаты для расчёта среднего.
print("Удаляемые строки")
print(df.loc[df["salary"].isna(), ["name", "department"]])
df = df.dropna(subset=["salary"]).copy()
df["total_comp"] = df["salary"] + df["bonus"]
print("\nПосле очистки")
print(df)

Удаляемые строки
   name department
8  Олег      Архив

После очистки
    name department   salary    bonus  total_comp
0   Анна         IT  95000.0  12000.0    107000.0
1   Иван         IT  75000.0      0.0     75000.0
2  Ольга         IT  85000.0   6000.0     91000.0
3  Павел    Продажи  90000.0  14000.0    104000.0
4  Елена    Продажи  70000.0   5000.0     75000.0
5  Ирина    Продажи  65000.0      0.0     65000.0
6  Денис         HR  68000.0   4000.0     72000.0
7   Алла         HR  68000.0   3000.0     71000.0


In [13]:
# //4.1 — Средняя зарплата по отделам
mean_salary = df.groupby("department")["salary"].mean()
print(mean_salary.round(2))

department
HR         68000.0
IT         85000.0
Продажи    75000.0
Name: salary, dtype: float64


In [14]:
# //4.2 — Средняя зарплата, максимальный бонус и численность
summary = df.groupby("department").agg(
    mean_salary=("salary", "mean"),
    max_bonus=("bonus", "max"),
    employees=("name", "size")
)
print(summary.round(2))

            mean_salary  max_bonus  employees
department                                   
HR              68000.0     4000.0          2
IT              85000.0    12000.0          3
Продажи         75000.0    14000.0          3


In [15]:
# //4.3 — Сводная таблица средних зарплат
pivot = df.pivot_table(index="department", values="salary", aggfunc="mean")
print(pivot.round(2))

             salary
department         
HR          68000.0
IT          85000.0
Продажи     75000.0


In [16]:
# //4.4 — Сортировка отделов по убыванию средней зарплаты
print(pivot.sort_values("salary", ascending=False).round(2))

             salary
department         
IT          85000.0
Продажи     75000.0
HR          68000.0


In [17]:
# //5.1 — Таблица руководителей отделов
managers = pd.DataFrame({
    "department": ["IT", "Продажи", "HR", "Архив"],
    "manager": ["Сергей", "Вера", "Мария", "Борис"]
})
print(managers)

  department manager
0         IT  Сергей
1    Продажи    Вера
2         HR   Мария
3      Архив   Борис


In [18]:
# //5.2 — Левое объединение по отделу
df = df.merge(managers, on="department", how="left")
print(df[["name", "department", "manager"]])

    name department manager
0   Анна         IT  Сергей
1   Иван         IT  Сергей
2  Ольга         IT  Сергей
3  Павел    Продажи    Вера
4  Елена    Продажи    Вера
5  Ирина    Продажи    Вера
6  Денис         HR   Мария
7   Алла         HR   Мария


In [19]:
# //5.4 — Сброс индекса после объединения
df = df.reset_index(drop=True)
print(df)

    name department   salary    bonus  total_comp manager
0   Анна         IT  95000.0  12000.0    107000.0  Сергей
1   Иван         IT  75000.0      0.0     75000.0  Сергей
2  Ольга         IT  85000.0   6000.0     91000.0  Сергей
3  Павел    Продажи  90000.0  14000.0    104000.0    Вера
4  Елена    Продажи  70000.0   5000.0     75000.0    Вера
5  Ирина    Продажи  65000.0      0.0     65000.0    Вера
6  Денис         HR  68000.0   4000.0     72000.0   Мария
7   Алла         HR  68000.0   3000.0     71000.0   Мария


In [20]:
# //6.1 — Три сотрудника с самой высокой общей компенсацией
top3 = df.nlargest(3, "total_comp")
print(top3[["name", "department", "total_comp"]].to_string(index=False))

 name department  total_comp
 Анна         IT    107000.0
Павел    Продажи    104000.0
Ольга         IT     91000.0


In [21]:
# //6.2 — Средняя зарплата и средний бонус в одной таблице
averages = df.groupby("department")[["salary", "bonus"]].mean()
print(averages.round(2))

             salary    bonus
department                  
HR          68000.0  3500.00
IT          85000.0  6000.00
Продажи     75000.0  6333.33


In [22]:
# //6.3 — Доля сотрудников с зарплатой строго выше 75 000
above_limit = df["salary"] > 75000
share = above_limit.groupby(df["department"]).mean()
print((share * 100).round(2).rename("Доля, %"))

department
HR          0.00
IT         66.67
Продажи    33.33
Name: Доля, %, dtype: float64
